
# Proposed Research Approach

The objective of this research is to investigate whether Genetic Algorithm (GA) feature selection and Particle Swarm Optimization (PSO) hyperparameter tuning can improve the predictive performance of ensemble machine learning models for heart disease detection while maintaining generalizability and interpretability.

This notebook will explore the following experiments:

### 1. Model Development and Optimization

I will apply GA-based feature selection and PSO-based hyperparameter optimization to two classification models:

- Soft Voting Ensemble
- Logistic Regression

I will evaluate the optimization techniques individually and in combination to determine whether they improve model performance.

The optimized models will be compared against their respective baseline models, which do not use GA or PSO.

Model performance will be evaluated using the following metrics:

- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC

### 2. Internal Testing and External Evaluation

I will initially train and test the models using real patient data. After establishing their internal performance, I will evaluate the finalized models using a simulated dataset to investigate how well they generalize to different data distributions.

I will examine differences in predictive performance between the real and simulated datasets.

### 3. Model Interpretability Using SHAP

I will apply SHAP (SHapley Additive exPlanations) to each model to investigate which features contribute most to its predictions.

I will compare feature importance across baseline and optimized models to determine whether GA and PSO influence which predictors the models rely on. I will also investigate whether important features remain consistent when evaluated on different datasets.

### 4. Early Heart Disease Risk Prediction

To further investigate the potential application of the proposed modeling approach in early risk identification, I will conduct a separate experiment using the Framingham Heart Disease dataset.

Unlike the previous datasets, Framingham uses a target variable representing whether an individual develops coronary heart disease (CHD) within ten years.

I will apply the same modeling and optimization approaches to this dataset to evaluate their effectiveness in predicting future CHD.

Finally, I will use SHAP to examine the features contributing to future CHD predictions and compare these findings with the existing heart disease detection experiment.

### Overall Research Goal

The overall goal is to determine whether GA and PSO optimization can improve ensemble model performance while maintaining a balance between predictive accuracy, generalizability, and interpretability across different heart disease prediction tasks.

# Importing Libraries & Settings

In [1]:
# %pip install -q numpy pandas scikit-learn imbalanced-learn matplotlib scipy joblib shap

import os, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import Parallel, delayed
from scipy.stats import spearmanr

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, roc_curve)
from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import sklearn

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 50)

# ---------- files ----------
UCI_PATH        = "uci_heart_disease.csv"       # internal dataset
KAGGLE_PATH     = "heart_disease_Kaggle.csv"    # external (simulated) dataset
FRAMINGHAM_PATH = "framingham.csv"              # Experiment 4 (skipped if the file is missing)
FRAMINGHAM_TARGET = "TenYearCHD"

# ---------- experiment settings ----------
SEEDS = [42]                  # add more (e.g. [42, 7, 123]) to get mean +/- std across splits
TEST_SIZE = 0.20
OVERSAMPLE = True             # random oversampling of the minority class, inside training folds only
FITNESS_SCORING = "f1"        # metric GA and PSO maximize (CV on training data)
FINAL_CV = 5                  # CV folds for the baseline grid searches
SEARCH_MAX_ROWS = 1500        # GA/PSO/grid search use at most this many training rows (speed)
N_JOBS = -1                   # parallel workers for candidate evaluation

GA_CFG  = dict(pop_size=10, generations=8, crossover_rate=0.8, mutation_rate=0.10, patience=4, cv=3)
PSO_CFG = dict(swarm_size=8, iterations=8, w=0.7, c1=1.5, c2=1.5, vmax=0.5, patience=4, cv=3)

# ---------- SHAP budget ----------
SHAP_BACKGROUND = 25          # background rows used to "remove" features
SHAP_EXPLAIN    = 60          # rows explained per model
SHAP_PERMS      = 4           # permutations per row (more = smoother, slower)

# Shared features used to compare UCI with Kaggle. Add "fbs" if the Kaggle column is usable (see Section 3).
SHARED_KEYS = ["age", "sex", "trestbps", "chol"]

MODELS  = ["Soft Voting", "Logistic Regression"]
CONFIGS = ["Baseline", "GA", "PSO", "GA+PSO"]

# Load Datasets

In [2]:
uci_raw     = pd.read_csv("UCI_heart_disease.csv")
kaggle_raw  = pd.read_csv("heart_disease_Kaggle.csv")
framingham_raw = pd.read_csv("framingham_preprocessed.csv")

print("UCI      :", uci_raw.shape)
print("Kaggle   :", kaggle_raw.shape)
print("Framingham:", None if framingham_raw is None else framingham_raw.shape)
print("\nUCI columns   :", list(uci_raw.columns))
print("Kaggle columns:", list(kaggle_raw.columns))
uci_raw.head()

UCI      : (918, 12)
Kaggle   : (10000, 21)
Framingham: (4133, 16)

UCI columns   : ['age', 'sex', 'chest pain type', 'resting bp s', 'cholesterol', 'fasting blood sugar', 'resting ecg', 'max heart rate', 'exercise angina', 'oldpeak', 'ST slope', 'target']
Kaggle columns: ['Age', 'Gender', 'Blood Pressure', 'Cholesterol Level', 'Exercise Habits', 'Smoking', 'Family Heart Disease', 'Diabetes', 'BMI', 'High Blood Pressure', 'Low HDL Cholesterol', 'High LDL Cholesterol', 'Alcohol Consumption', 'Stress Level', 'Sleep Hours', 'Sugar Consumption', 'Triglyceride Level', 'Fasting Blood Sugar', 'CRP Level', 'Homocysteine Level', 'Heart Disease Status']


,age,sex,chest pain type,resting bp s,cholesterol,fasting blood sugar,resting ecg,max heart rate,exercise angina,oldpeak,ST slope,target
0,40,1,2,140,289,0,0,172,0,0.0,1,0
1,49,0,3,160,180,0,0,156,0,1.0,2,1
2,37,1,2,130,283,0,1,98,0,0.0,1,0
3,48,0,4,138,214,0,0,108,1,1.5,2,1
4,54,1,3,150,195,0,0,122,0,0.0,1,0


Cleans Targets & Makes Columns Consistent

In [6]:
ALIASES = {
    "age":      ["age"],
    "sex":      ["sex", "gender", "male"],
    "trestbps": ["restingbp", "trestbps", "resting_bp", "blood pressure", "bloodpressure", "sysbp", "resting bp s", "resting bp"],
    "chol":     ["cholesterol", "chol", "cholesterol level", "totchol"],
    "fbs":      ["fastingbs", "fbs", "fasting blood sugar"],
}
TARGET_ALIASES = ["target", "heartdisease", "heart_disease", "heart disease status", "num", "output", "condition"]

def find_col(df, aliases):
    lookup = {c.lower().strip(): c for c in df.columns}
    for a in aliases:
        if a in lookup:
            return lookup[a]
    return None

def canon_name(col):
    # Map a column name to its canonical key (e.g. 'RestingBP' -> 'trestbps'); unknown names stay unchanged.
    for key, al in ALIASES.items():
        if col.lower().strip() in al:
            return key
    return col

def to_binary(s):
    if pd.api.types.is_numeric_dtype(s):
        return (s > 0).astype(int)
    return s.astype(str).str.strip().str.lower().isin(["yes", "y", "true", "1", "present", "disease"]).astype(int)

def load_xy(df, target_aliases):
    tcol = find_col(df, target_aliases)
    assert tcol is not None, f"No target column found. Columns: {list(df.columns)}"
    df = df.dropna(subset=[tcol]).reset_index(drop=True)
    y = to_binary(df[tcol])
    X = df.drop(columns=[tcol])
    X = X.drop(columns=[c for c in X.columns if c.lower() in ("id", "unnamed: 0")])
    for key in ("trestbps", "chol"):                    # 0 is not a valid reading -> missing
        c = find_col(X, ALIASES[key])
        if c is not None:
            X[c] = X[c].replace(0, np.nan)
    return X, y

def encode_sex(s):
    if pd.api.types.is_numeric_dtype(s):
        return s.astype(float)
    t = s.astype(str).str.strip().str.lower()
    out = t.str.startswith("m").astype(float)
    return out.where(s.notna(), np.nan)

def to_shared(X):
    # Keep only the features every dataset has, with common names and units.
    out = {}
    for k in SHARED_KEYS:
        c = find_col(X, ALIASES[k])
        if c is None:
            raise KeyError(f"Shared feature '{k}' not found; columns: {list(X.columns)}")
        s = X[c]
        if k == "sex":
            s = encode_sex(s)
        s = pd.to_numeric(s, errors="coerce")
        if k == "fbs" and s.dropna().max() > 1:         # continuous glucose -> '>120 mg/dl' flag
            s = (s > 120).astype(float).where(s.notna(), np.nan)
        out[k] = s
    return pd.DataFrame(out)

def encode_frame(X):
    # Text columns -> integer codes (float, NaN kept). Returns the frame and {'num': [...], 'cat': [...]}.
    X = X.copy()
    num, cat = [], []
    for c in X.columns:
        if not pd.api.types.is_numeric_dtype(X[c]):
            codes = pd.Categorical(X[c]).codes.astype(float)
            X[c] = np.where(codes < 0, np.nan, codes)
            cat.append(c)
        else:
            X[c] = X[c].astype(float)
            nun = X[c].nunique(dropna=True)
            is_int = np.allclose(X[c].dropna() % 1, 0)
            (cat if (2 < nun <= 5 and is_int) else num).append(c)
    return X, {"num": num, "cat": cat}

# ---- UCI (internal) ----
X_uci_raw, y_uci = load_xy(uci_raw, TARGET_ALIASES)
X_uci_full, types_uci_full = encode_frame(X_uci_raw)

# ---- Kaggle (external, simulated) ----
X_kag_raw, y_kag = load_xy(kaggle_raw, ["heart disease status"] + TARGET_ALIASES)

# ---- shared-feature views ----
X_uci_sh = to_shared(X_uci_raw)
X_kag_sh = to_shared(X_kag_raw)
types_shared = {"num": list(X_uci_sh.columns), "cat": []}

print(f"UCI   : {X_uci_full.shape}, disease prevalence {y_uci.mean():.1%}")
print(f"Kaggle: {X_kag_raw.shape}, disease prevalence {y_kag.mean():.1%}")
print("\nUCI numeric:", types_uci_full['num'], "\nUCI categorical:", types_uci_full['cat'])
print("\nShared-feature summary (check that the units line up!):")
summary = pd.concat({"UCI": X_uci_sh.describe().T[["mean", "std", "min", "max"]],
                     "Kaggle": X_kag_sh.describe().T[["mean", "std", "min", "max"]]}, axis=1)
summary

UCI   : (918, 11), disease prevalence 55.3%
Kaggle: (10000, 20), disease prevalence 20.0%

UCI numeric: ['age', 'sex', 'resting bp s', 'cholesterol', 'fasting blood sugar', 'max heart rate', 'exercise angina', 'oldpeak'] 
UCI categorical: ['chest pain type', 'resting ecg', 'ST slope']

Shared-feature summary (check that the units line up!):


UCI                              Kaggle                         
                mean        std   min    max        mean        std    min    max
age        53.510893   9.432617  28.0   77.0   49.296259  18.193970   18.0   80.0
sex         0.789760   0.407701   0.0    1.0    0.501252   0.500023    0.0    1.0
trestbps  132.540894  17.999749  80.0  200.0  149.757740  17.572969  120.0  180.0
chol      244.635389  59.153524  85.0  603.0  225.425577  43.575809  150.0  300.0

# Preprocessing Pipeline

- Applies GA
- Scales or one hot encoding
- Oversamples training data only
- Fits Classifier
- Builds SVM + MLP + Extra Trees
  - Logistic Regression

In [7]:
def build_pipeline(clf, cols, types, seed):
    num = [c for c in cols if c in types["num"]]
    cat = [c for c in cols if c in types["cat"]]
    tf = []
    if num:
        tf.append(("num", SkPipeline([("imp", SimpleImputer(strategy="median")),
                                      ("sc", StandardScaler())]), num))
    if cat:
        tf.append(("cat", SkPipeline([("imp", SimpleImputer(strategy="most_frequent")),
                                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat))
    prep = ColumnTransformer(tf, remainder="drop")
    ros = RandomOverSampler(random_state=seed) if OVERSAMPLE else "passthrough"
    return ImbPipeline([("prep", prep), ("ros", ros), ("clf", clf)])

def make_lr(C=1.0, l1_ratio=None, seed=42):
    if l1_ratio is None:                       # plain L2 (grid-search baseline)
        return LogisticRegression(C=C, max_iter=5000)
    kw = dict(C=C, l1_ratio=float(l1_ratio), solver="saga", max_iter=5000, random_state=seed)
    major, minor = (int(v) for v in sklearn.__version__.split(".")[:2])
    if (major, minor) < (1, 8):                # newer versions infer elastic-net from l1_ratio
        kw["penalty"] = "elasticnet"
    return LogisticRegression(**kw)

def make_sve(p, seed=42, search=False):
    u = int(p["mlp_units"])
    svm = SVC(kernel="rbf", C=p["svm_C"], gamma=p["svm_gamma"], probability=True, random_state=seed)
    mlp = MLPClassifier(activation="relu", learning_rate="adaptive", alpha=p["mlp_alpha"],
                        hidden_layer_sizes=(u, max(u // 2, 2)),
                        max_iter=300 if search else 500, random_state=seed)
    et  = ExtraTreesClassifier(n_estimators=int(p["et_n"]), max_depth=int(p["et_depth"]),
                               min_samples_leaf=int(p["et_leaf"]), random_state=seed, n_jobs=1)
    return VotingClassifier([("svm", svm), ("mlp", mlp), ("et", et)], voting="soft")

def make_model(name, params, seed=42, search=False):
    if name == "Logistic Regression":
        return make_lr(params["C"], params.get("l1_ratio"), seed)
    return make_sve(params, seed, search)

# Moderate defaults, used as the fixed model while GA scores feature subsets
DEFAULT_PARAMS = {
    "Logistic Regression": {"C": 1.0},
    "Soft Voting": dict(svm_C=10, svm_gamma=0.1, mlp_alpha=1e-3, mlp_units=64,
                        et_n=100, et_depth=10, et_leaf=1),
}

Fitness Function & Genetic Algorithm (Parent Paper)
- Fitness = cross validates score of a model on a feature subset (Only training data)
- Genetic Algorithm: Uses roulette-wheel selection, one point crossover, bit flip mutation, and replaces the worst half of each generation.

In [8]:
def cv_fitness(model_name, params, cols, X, y, types, cv_splits, seed):
    if len(cols) == 0:
        return 0.0
    pipe = build_pipeline(make_model(model_name, params, seed, search=True), cols, types, seed)
    skf = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=seed)
    try:
        return float(cross_val_score(pipe, X, y, cv=skf, scoring=FITNESS_SCORING).mean())
    except Exception:
        return 0.0

def ga_select(model_name, X, y, types, seed, cfg=GA_CFG, verbose=True):
    rng = np.random.default_rng(seed)
    feats = list(X.columns); n = len(feats)
    if n < 2:
        return feats, np.nan, []
    P = cfg["pop_size"]
    pop = rng.integers(0, 2, size=(P, n))
    for i in range(P):
        if pop[i].sum() == 0:
            pop[i, rng.integers(n)] = 1
    cache = {}

    def evaluate(popu):
        keys = [tuple(int(v) for v in r) for r in popu]
        todo = [k for k in dict.fromkeys(keys) if k not in cache]
        res = Parallel(n_jobs=N_JOBS)(
            delayed(cv_fitness)(model_name, DEFAULT_PARAMS[model_name],
                                [f for f, b in zip(feats, k) if b], X, y, types, cfg["cv"], seed)
            for k in todo)
        cache.update(zip(todo, res))
        return np.array([cache[k] for k in keys])

    best_key, best_chrom, best_fit, stale, history = None, None, -np.inf, 0, []
    for gen in range(1, cfg["generations"] + 1):
        fit = evaluate(pop)
        # best of generation: highest fitness, then fewest features
        order = sorted(range(P), key=lambda i: (round(fit[i], 4), -pop[i].sum()), reverse=True)
        i0 = order[0]; key = (round(fit[i0], 4), -int(pop[i0].sum()))
        if best_key is None or key > best_key:
            best_key, best_chrom, best_fit, stale = key, pop[i0].copy(), fit[i0], 0
        else:
            stale += 1
        history.append(best_fit)
        if verbose:
            print(f"  [GA] gen {gen:02d} | best {FITNESS_SCORING} = {best_fit:.4f} | features = {int(best_chrom.sum())}/{n}")
        if stale >= cfg["patience"]:
            break

        shifted = fit - fit.min() + 1e-6                               # roulette-wheel selection
        parents = pop[rng.choice(P, size=P, p=shifted / shifted.sum())]
        child = parents.copy()
        for i in range(0, P - 1, 2):                                   # one-point crossover
            if rng.random() < cfg["crossover_rate"]:
                pt = rng.integers(1, n)
                child[i, pt:], child[i + 1, pt:] = child[i + 1, pt:].copy(), child[i, pt:].copy()
        flip = rng.random(child.shape) < cfg["mutation_rate"]          # bit-flip mutation
        child[flip] = 1 - child[flip]
        for i in range(P):
            if child[i].sum() == 0:
                child[i, rng.integers(n)] = 1
        worst = np.argsort(fit)[: P // 2]                              # replace the worst half
        pop[worst] = child[: P // 2]

    selected = [f for f, b in zip(feats, best_chrom) if b]
    return selected, best_fit, history

# Particle Swarm Optimization
Particles are hyperparameter vectors in a [0,1] box (normalized)
  - Decoded with log scaling for C, gamma and alpha
  - Rounding for integer parameters
  

In [9]:
PSO_SPECS = {
    "Soft Voting": [("svm_C", "log", (0.1, 100)), ("svm_gamma", "log", (1e-3, 1.0)),
                    ("mlp_alpha", "log", (1e-5, 1e-1)), ("mlp_units", "int", (16, 128)),
                    ("et_n", "int", (50, 300)), ("et_depth", "int", (3, 20)), ("et_leaf", "int", (1, 5))],
    "Logistic Regression": [("C", "log", (1e-3, 1e2)), ("l1_ratio", "lin", (0.0, 1.0))],
}

def decode(u, specs):
    out = {}
    for ui, (name, kind, (lo, hi)) in zip(u, specs):
        if kind == "log":
            v = 10 ** (np.log10(lo) + ui * (np.log10(hi) - np.log10(lo)))
        elif kind == "int":
            v = int(round(lo + ui * (hi - lo)))
        else:
            v = lo + ui * (hi - lo)
        out[name] = v
    return out

def pso_tune(model_name, X, y, cols, types, seed, cfg=PSO_CFG, verbose=True):
    rng = np.random.default_rng(seed)
    specs = PSO_SPECS[model_name]; d = len(specs); N = cfg["swarm_size"]

    def evaluate(P):
        return np.array(Parallel(n_jobs=N_JOBS)(
            delayed(cv_fitness)(model_name, decode(u, specs), cols, X, y, types, cfg["cv"], seed) for u in P))

    pos = rng.random((N, d)); vel = np.zeros((N, d))
    pbest, pbest_fit = pos.copy(), evaluate(pos)
    g = int(np.argmax(pbest_fit)); gpos, gfit = pbest[g].copy(), pbest_fit[g]
    history, stale = [gfit], 0
    if verbose:
        print(f"  [PSO] init | best {FITNESS_SCORING} = {gfit:.4f}")
    for it in range(1, cfg["iterations"] + 1):
        r1, r2 = rng.random((N, d)), rng.random((N, d))
        vel = cfg["w"] * vel + cfg["c1"] * r1 * (pbest - pos) + cfg["c2"] * r2 * (gpos - pos)
        vel = np.clip(vel, -cfg["vmax"], cfg["vmax"])
        pos = np.clip(pos + vel, 0.0, 1.0)
        fit = evaluate(pos)
        better = fit > pbest_fit
        pbest[better], pbest_fit[better] = pos[better], fit[better]
        g = int(np.argmax(pbest_fit))
        if pbest_fit[g] > gfit:
            gfit, gpos, stale = pbest_fit[g], pbest[g].copy(), 0
        else:
            stale += 1
        history.append(gfit)
        if verbose:
            print(f"  [PSO] iter {it:02d} | best {FITNESS_SCORING} = {gfit:.4f}")
        if stale >= cfg["patience"]:
            break
    return decode(gpos, specs), gfit, history

# Baseline Hyperparameter Tuning: Grid Search (Paper 2)

Uses grids for ensemble method & Logistic Regression

In [10]:
def grid_tune(model_name, X, y, cols, types, seed):
    cv = StratifiedKFold(n_splits=FINAL_CV, shuffle=True, random_state=seed)
    def gs(clf, grid):
        g = GridSearchCV(build_pipeline(clf, cols, types, seed), grid, cv=cv,
                         scoring=FITNESS_SCORING, n_jobs=N_JOBS)
        return g.fit(X, y).best_params_
    if model_name == "Logistic Regression":
        b = gs(LogisticRegression(max_iter=5000), {"clf__C": [0.01, 0.1, 1, 10, 100]})
        return {"C": b["clf__C"]}
    b1 = gs(SVC(kernel="rbf", probability=False, random_state=seed),
            {"clf__C": [1, 10], "clf__gamma": [0.01, 0.1]})
    b2 = gs(MLPClassifier(activation="relu", learning_rate="adaptive", max_iter=500, random_state=seed),
            {"clf__alpha": [1e-4, 1e-3], "clf__hidden_layer_sizes": [(64, 32), (128, 64)]})
    b3 = gs(ExtraTreesClassifier(random_state=seed, n_jobs=1),
            {"clf__n_estimators": [100, 200], "clf__max_depth": [10, 20], "clf__min_samples_leaf": [1, 3]})
    return dict(svm_C=b1["clf__C"], svm_gamma=b1["clf__gamma"],
                mlp_alpha=b2["clf__alpha"], mlp_units=b2["clf__hidden_layer_sizes"][0],
                et_n=b3["clf__n_estimators"], et_depth=b3["clf__max_depth"], et_leaf=b3["clf__min_samples_leaf"])

# Evaluation Metrics & Model Run

 run_study:
 - Train/test split
 - Runs each model:
  - Baseline
  - GA
  - PSO
  - GA + PSO
- only on training data then scores on the internal test split & external dataset

In [11]:
def score_all(y, proba, thr=0.5):
    pred = (proba >= thr).astype(int)
    return {"Accuracy":  100 * accuracy_score(y, pred),
            "Precision": 100 * precision_score(y, pred, zero_division=0),
            "Recall":    100 * recall_score(y, pred, zero_division=0),
            "F1":        100 * f1_score(y, pred, zero_division=0),
            "ROC-AUC":   100 * roc_auc_score(y, proba) if len(np.unique(y)) > 1 else np.nan}

def search_subset(X, y, seed):
    if len(X) <= SEARCH_MAX_ROWS:
        return X, y
    Xs, _, ys, _ = train_test_split(X, y, train_size=SEARCH_MAX_ROWS, stratify=y, random_state=seed)
    return Xs, ys

def run_study(study, X, y, types, externals=None, seed=42, verbose=True):
    externals = externals or {}
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=seed)
    Xtr, Xte = Xtr.reset_index(drop=True), Xte.reset_index(drop=True)
    ytr, yte = ytr.reset_index(drop=True), yte.reset_index(drop=True)
    Xs, ys = search_subset(Xtr, ytr, seed)
    all_cols = list(X.columns)
    rows, fitted, info = [], {}, {}

    for m in MODELS:
        if verbose: print(f"\n=== {study} | {m} ===")
        t0 = time.time()
        if verbose: print(" Baseline: grid search on all features")
        base_params = grid_tune(m, Xs, ys, all_cols, types, seed)
        if verbose: print(" GA: feature selection")
        ga_cols, ga_fit, ga_hist = ga_select(m, Xs, ys, types, seed, verbose=verbose)
        if verbose: print(" GA: grid search on selected features")
        ga_params = grid_tune(m, Xs, ys, ga_cols, types, seed)
        if verbose: print(" PSO: tuning on all features")
        pso_params, pso_fit, pso_hist = pso_tune(m, Xs, ys, all_cols, types, seed, verbose=verbose)
        if verbose: print(" GA+PSO: tuning on selected features")
        gp_params, gp_fit, gp_hist = pso_tune(m, Xs, ys, ga_cols, types, seed, verbose=verbose)

        plan = {"Baseline": (all_cols, base_params), "GA": (ga_cols, ga_params),
                "PSO": (all_cols, pso_params),       "GA+PSO": (ga_cols, gp_params)}
        info[m] = {"ga_history": ga_hist, "pso_history": pso_hist, "gapso_history": gp_hist,
                   "ga_features": ga_cols}
        for cfg_name, (cols, params) in plan.items():
            pipe = build_pipeline(make_model(m, params, seed), cols, types, seed).fit(Xtr, ytr)
            fitted[(m, cfg_name)] = (pipe, cols, params)
            evals = {f"{study} (internal test)": (Xte, yte)}
            evals.update(externals)
            for ds_name, (Xe, ye) in evals.items():
                proba = pipe.predict_proba(Xe[all_cols])[:, 1]
                rows.append({"Study": study, "Model": m, "Config": cfg_name, "Dataset": ds_name,
                             "Seed": seed, **score_all(ye, proba),
                             "n_features": len(cols), "features": ", ".join(cols),
                             "params": {k: (round(v, 5) if isinstance(v, float) else v) for k, v in params.items()}})
        if verbose: print(f" done in {time.time() - t0:.0f}s | GA kept {len(ga_cols)}/{len(all_cols)}: {ga_cols}")
    return {"results": pd.DataFrame(rows), "fitted": fitted, "info": info,
            "split": (Xtr, Xte, ytr, yte), "all_cols": all_cols, "types": types}

def run_seeds(study, X, y, types, externals=None):
    # Runs the study once per seed. SHAP/ROC plots use the first seed's fitted models.
    runs = [run_study(study, X, y, types, externals, seed=s, verbose=(i == 0)) for i, s in enumerate(SEEDS)]
    runs[0]["results"] = pd.concat([r["results"] for r in runs], ignore_index=True)
    return runs[0]

METRICS = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]
def summarize(res, dataset):
    d = res[res["Dataset"] == dataset]
    g = d.groupby(["Model", "Config"], sort=False)
    out = g[METRICS].mean().round(2)
    if len(SEEDS) > 1:
        out = out.astype(str) + " ± " + g[METRICS].std().round(2).astype(str)
    out["n_features"] = g["n_features"].mean().round(1)
    return out.reindex(pd.MultiIndex.from_product([MODELS, CONFIGS]))

def delta_vs_baseline(res, dataset):
    d = res[res["Dataset"] == dataset].groupby(["Model", "Config"], sort=False)[METRICS].mean()
    base = d.xs("Baseline", level="Config")
    return (d - base.reindex(d.index.get_level_values("Model")).values).round(2).reindex(
        pd.MultiIndex.from_product([MODELS, CONFIGS]))

# Experiment 1: UCI with all features

In [ ]:
study1 = run_seeds("UCI (all features)", X_uci_full, y_uci, types_uci_full)
ds1 = "UCI (all features) (internal test)"
summarize(study1["results"], ds1)

In [ ]:
# Change relative to baseline (positive means optimization helped)
delta_vs_baseline(study1["results"], ds1)

In [ ]:
# Features the GA kept and search convergence
for m in MODELS:
    kept = study1["info"][m]["ga_features"]
    dropped = [c for c in study1["all_cols"] if c not in kept]
    print(f"{m}: kept {len(kept)}/{len(study1['all_cols'])} -> {kept}\n   dropped: {dropped}")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
for ax, m in zip(axes, MODELS):
    inf = study1["info"][m]
    ax.plot(inf["ga_history"], marker="o", label="GA (best fitness)")
    ax.plot(inf["pso_history"], marker="s", label="PSO, all features")
    ax.plot(inf["gapso_history"], marker="^", label="PSO, GA features")
    ax.set_title(m); ax.set_xlabel("generation / iteration"); ax.set_ylabel(f"CV {FITNESS_SCORING}")
    ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Metric Bar Charts & ROC Curves
def plot_metric_bars(res, dataset, title):
    d = res[res["Dataset"] == dataset].groupby(["Model", "Config"], sort=False)[METRICS].mean()
    fig, axes = plt.subplots(1, len(MODELS), figsize=(12, 3.8), sharey=True)
    w = 0.8 / len(CONFIGS)
    for ax, m in zip(np.atleast_1d(axes), MODELS):
        for i, cfg in enumerate(CONFIGS):
            ax.bar(np.arange(len(METRICS)) + i * w, d.loc[(m, cfg)].values, w, label=cfg)
        ax.set_xticks(np.arange(len(METRICS)) + 1.5 * w); ax.set_xticklabels(METRICS, rotation=20)
        ax.set_title(m); ax.set_ylim(max(0, d.values.min() - 10), 100)
    np.atleast_1d(axes)[0].set_ylabel("%"); np.atleast_1d(axes)[0].legend(fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

def plot_rocs(study, title):
    Xtr, Xte, ytr, yte = study["split"]
    fig, axes = plt.subplots(1, len(MODELS), figsize=(10, 4))
    for ax, m in zip(np.atleast_1d(axes), MODELS):
        for cfg in CONFIGS:
            pipe, cols, _ = study["fitted"][(m, cfg)]
            p = pipe.predict_proba(Xte[study["all_cols"]])[:, 1]
            fpr, tpr, _ = roc_curve(yte, p)
            ax.plot(fpr, tpr, label=f"{cfg} (AUC {roc_auc_score(yte, p):.3f})")
        ax.plot([0, 1], [0, 1], "k--", alpha=.4); ax.set_title(m)
        ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate"); ax.legend(fontsize=8)
    fig.suptitle(title); plt.tight_layout(); plt.show()

plot_metric_bars(study1["results"], ds1, "Experiment 1: UCI internal test performance")
plot_rocs(study1, "Experiment 1: ROC curves (UCI internal test)")

# Experiment 2: Internal vs External Evaluation

- Retrains on shared features only
- Tests on the UCI hold-out
- Then evaluates on the whole Kaggle dataset (never used for training, feature selection or tuning)

In [ ]:
study2 = run_seeds("UCI (shared features)", X_uci_sh, y_uci, types_shared,
                   externals={"Kaggle (external)": (X_kag_sh, y_kag)})
ds_int, ds_ext = "UCI (shared features) (internal test)", "Kaggle (external)"
print("INTERNAL (UCI hold-out)"); display(summarize(study2["results"], ds_int))
print("EXTERNAL (Kaggle simulated)"); display(summarize(study2["results"], ds_ext))

In [ ]:
# Internal vs External Gap
# ROC-AUC is the Fairest Column
r = study2["results"]
gap = (r[r.Dataset == ds_ext].groupby(["Model", "Config"], sort=False)[METRICS].mean()
       - r[r.Dataset == ds_int].groupby(["Model", "Config"], sort=False)[METRICS].mean()).round(2)
print(f"Disease prevalence  UCI: {y_uci.mean():.1%} | Kaggle: {y_kag.mean():.1%}")
display(gap.reindex(pd.MultiIndex.from_product([MODELS, CONFIGS])))
plot_metric_bars(r, ds_ext, "Experiment 2: external (Kaggle) performance")

d = r.groupby(["Model", "Config", "Dataset"], sort=False)["ROC-AUC"].mean().unstack("Dataset")
d = d.reindex(pd.MultiIndex.from_product([MODELS, CONFIGS]))
d.plot(kind="bar", figsize=(10, 3.8)); plt.ylabel("ROC-AUC (%)"); plt.axhline(50, color="k", ls=":", lw=1)
plt.title("ROC-AUC: internal vs external (dotted line = chance)"); plt.xticks(rotation=30, ha="right")
plt.tight_layout(); plt.show()

# Experiment 3: SHAP
- Runs on the entire pipelines predicted probability using original features as inputs

- A feature GA dropped gets SHAP = 0
  - Importance is mean absolute SHAP normalized to sum to 1


In [ ]:
def shap_importance(pipe, all_cols, X_bg, X_explain, seed=42):
    import shap
    def f(arr):
        return pipe.predict_proba(pd.DataFrame(np.asarray(arr, dtype=float), columns=all_cols))[:, 1]
    bg = X_bg[all_cols].sample(min(SHAP_BACKGROUND, len(X_bg)), random_state=seed)
    ex = X_explain[all_cols].sample(min(SHAP_EXPLAIN, len(X_explain)), random_state=seed)
    masker = shap.maskers.Independent(bg, max_samples=len(bg))
    explainer = shap.Explainer(f, masker, algorithm="permutation", seed=seed)
    sv = explainer(ex, max_evals=2 * len(all_cols) * SHAP_PERMS + 1)
    imp = pd.Series(np.abs(sv.values).mean(axis=0), index=all_cols)
    return imp / imp.sum() if imp.sum() > 0 else imp

def shap_table(study, X_explain, label, seed=42):
    # Returns {(model, config): normalized importance Series} for one dataset.
    Xtr = study["split"][0]; out = {}
    for m in MODELS:
        for cfg in CONFIGS:
            pipe = study["fitted"][(m, cfg)][0]
            out[(m, cfg)] = shap_importance(pipe, study["all_cols"], Xtr, X_explain, seed)
        print(f"  SHAP done: {label} | {m}")
    return out

def plot_importance(tab, title, top=10):
    fig, axes = plt.subplots(1, len(MODELS), figsize=(13, 4.5))
    for ax, m in zip(np.atleast_1d(axes), MODELS):
        df = pd.DataFrame({cfg: tab[(m, cfg)] for cfg in CONFIGS})
        df = df.loc[df.mean(axis=1).sort_values(ascending=False).index[:top]].iloc[::-1]
        df.plot(kind="barh", ax=ax, width=0.8); ax.set_title(m); ax.set_xlabel("share of mean |SHAP|")
    fig.suptitle(title); plt.tight_layout(); plt.show()

def rank_agreement(tab, ref=("Baseline",)):
    # Spearman correlation of feature importances between Baseline and each optimized config, plus top-3 lists.
    rows = []
    for m in MODELS:
        base = tab[(m, "Baseline")]
        for cfg in CONFIGS:
            s = tab[(m, cfg)]
            rho = spearmanr(base, s)[0] if s.nunique() > 1 and base.nunique() > 1 else np.nan
            rows.append({"Model": m, "Config": cfg, "Spearman vs Baseline": round(rho, 3),
                         "Top-3 features": ", ".join(s.sort_values(ascending=False).index[:3])})
    return pd.DataFrame(rows).set_index(["Model", "Config"])

Baseline vs Optimized
- shared featured models, on the UCI internal test split

In [ ]:
Xte2 = study2["split"][1]
shap_int = shap_table(study2, Xte2, "UCI internal")
plot_importance(shap_int, "SHAP importance on UCI internal test: baseline vs optimized")
display(rank_agreement(shap_int))

Same Models on the Kaggle External Data

- high correlation = model relies on the same predictors in both datasets


In [ ]:
shap_ext = shap_table(study2, X_kag_sh, "Kaggle external")
plot_importance(shap_ext, "SHAP importance on Kaggle external data")

rows = []
for m in MODELS:
    for cfg in CONFIGS:
        a, b = shap_int[(m, cfg)], shap_ext[(m, cfg)]
        rho = spearmanr(a, b)[0] if a.nunique() > 1 and b.nunique() > 1 else np.nan
        rows.append({"Model": m, "Config": cfg, "Spearman internal vs external": round(rho, 3),
                     "Top-3 internal": ", ".join(a.sort_values(ascending=False).index[:3]),
                     "Top-3 external": ", ".join(b.sort_values(ascending=False).index[:3])})
pd.DataFrame(rows).set_index(["Model", "Config"])

SHAP for all features on UCI models

In [ ]:
shap_full = shap_table(study1, study1["split"][1], "UCI all features")
plot_importance(shap_full, "SHAP importance, UCI all features (internal test)")
display(rank_agreement(shap_full))

In [ ]:
# Save Results
all_res = [study1["results"], study2["results"]]
if framingham_raw is not None:
    all_res.append(study4["results"])
final = pd.concat(all_res, ignore_index=True)
final.to_csv("ga_pso_results.csv", index=False)
print("Saved ga_pso_results.csv:", final.shape)
final[["Study", "Dataset", "Model", "Config", "ROC-AUC", "F1", "n_features", "features", "params"]].head(12)